In this notebook, we do inference on the VANCOUVER dataset, with the SPT model trained on the DALES dataset. These datasets are outdoor datasets with common classes. We have done a class matching to avoid issues.

In [1]:
import os
import sys

file_path = os.path.dirname(os.path.abspath(''))  # for .ipynb notebook
sys.path.append(file_path)

import laspy
import torch
from src.data import Data
from src.utils.color import to_float_rgb


def read_vancouver_tile(
        filepath, 
        xyz=True, 
        rgb=True, 
        intensity=True, 
        semantic=True, 
        instance=False,
        remap=True, 
        max_intensity=600):
    """Read a Vancouver tile saved as LAS.

    :param filepath: str
        Absolute path to the LAS file
    :param xyz: bool
        Whether XYZ coordinates should be saved in the output Data.pos
    :param rgb: bool
        Whether RGB colors should be saved in the output Data.rgb
    :param intensity: bool
        Whether intensity should be saved in the output Data.rgb
    :param semantic: bool
        Whether semantic labels should be saved in the output Data.y
    :param instance: bool
        Whether instance labels should be saved in the output Data.obj
    :param remap: bool
        Whether semantic labels should be mapped from their Vancouver ID
        to their train ID
    :param max_intensity: float
        Maximum value used to clip intensity signal before normalizing 
        to [0, 1]
    """
    # Create an emty Data object
    data = Data()
    
    las = laspy.read(filepath)

    # Populate data with point coordinates 
    if xyz:
        # Apply the scale provided by the LAS header
        pos = torch.stack([
            torch.tensor(las[axis])
            for axis in ["X", "Y", "Z"]], dim=-1)
        pos *= las.header.scale
        pos_offset = pos[0]
        data.pos = (pos - pos_offset).float()
        data.pos_offset = pos_offset

    # Populate data with point RGB colors
    if rgb:
        # RGB stored in uint16 lives in [0, 65535]
        data.rgb = to_float_rgb(torch.stack([
            torch.FloatTensor(las[axis].astype('float32') / 65535)
            for axis in ["red", "green", "blue"]], dim=-1))

    # Populate data with point LiDAR intensity
    if intensity:
        # Heuristic to bring the intensity distribution in [0, 1]
        data.intensity = torch.FloatTensor(
            las['intensity'].astype('float32')
        ).clip(min=0, max=max_intensity) / max_intensity

    # Populate data with point semantic segmentation labels
    if semantic:
        y = torch.LongTensor(las['classification'])
        data.y = torch.from_numpy(ID2TRAINID)[y] if remap else y

    # Populate data with point panoptic segmentation labels
    if instance:
        raise NotImplementedError("The dataset does not contain instance labels.")

    return data


import numpy as np

# Number of classes in the dataset (excluding void/unlabeled/ignored)
VANCOUVER_NUM_CLASSES = 6

# Mapping from original classes
ID2TRAINID = np.asarray([
    VANCOUVER_NUM_CLASSES,  # 0 Not used         ->  6 Ignored
    5,                      # 1 Other            ->  5 Other
    0,                      # 2 Ground           ->  0 Ground
    3,                      # 3 Low vegetation   ->  3 Low vegetation
    VANCOUVER_NUM_CLASSES,  # 4 Unknown / Noise  ->  6 Ignored
    2,                      # 5 High vegetation  ->  2 High vegetation
    4,                      # 6 Building         ->  4 Buildings
    VANCOUVER_NUM_CLASSES,  # 7 Unknown / Noise  ->  6 Ignored
    VANCOUVER_NUM_CLASSES,  # 8 Unknown / Noise  ->  6 Ignored
    1])                     # 9 Water            ->  1 Water

# Class names (including void/unlabeled/ignored last)
VANCOUVER_CLASS_NAMES = [
    'Ground',
    'Water',
    'High vegetation',
    'Low vegetation',
    'Buildings',
    'Other',
    'Ignored']

# Class color palette (including void/unlabeled/ignored last)
VANCOUVER_CLASS_COLORS = np.asarray([
    [243, 214, 171],
    [169, 222, 249],
    [ 70, 115,  66],
    [204, 213, 174],
    [214,  66,  54],
    [186, 160, 164],
    [  0,   0,   0]])

In [2]:
filepath = './482000_5455000.las'
data = read_vancouver_tile(filepath)

In [ ]:
data.show(class_names=VANCOUVER_CLASS_NAMES, class_colors=VANCOUVER_CLASS_COLORS)

number of points in our 1km^2 square

In [3]:
data.num_points

60353793

In [4]:
from src.transforms import SampleXYTiling, GridSampling3D

voxelized_data = GridSampling3D(0.5)(data) #voxelization to 50cm

In [ ]:
voxelized_data.show(class_names=VANCOUVER_CLASS_NAMES, class_colors=VANCOUVER_CLASS_COLORS)

After voxelization,we have now fewer points in our cloud:

In [5]:
voxelized_data.num_points

20198485

Tiling

In [ ]:
from src.transforms import SampleXYTiling, GridSampling3D
from src.data import Batch

# Tile the cloud into `xy_tiling` XY-oriented chunks of equal horizontal 
# span
xy_tiling = (3,3)


# Compute each chunk 
chunks = []
for x in range(xy_tiling[0]):
    for y in range(xy_tiling[1]):        
        # Extract the chunk at (x, y) in the tiling grid
        chunk = SampleXYTiling(x=x, y=y, tiling=xy_tiling)(voxelized_data)

        # Add a 'tile' attribute to the points for visualization
        chunk.tile = torch.full((chunk.num_points,), x * xy_tiling[1] + y)
        
        # Store the chunk for later aggregation
        chunks.append(chunk)

# Aggregate all chunk `Data` objects into one big `Data` object
data_tiled = Batch.from_data_list(chunks)

# Show the resulting `Data' with the 'tile' attribute
data_tiled.show(keys='tile')

In [6]:
# Extract the chunk at (x, y) in the tiling grid 3 by 3
tiled_data = SampleXYTiling(x=1, y=1, tiling=3)(voxelized_data) #the middle one

In a tile of the global square we also have fewer points

In [7]:
tiled_data.num_points

3136464

In [ ]:
tiled_data.show(class_names=VANCOUVER_CLASS_NAMES, class_colors=VANCOUVER_CLASS_COLORS)

Here we load the pre-processing config for used to train the SPT model on DALES dataset

In [8]:
from src.utils import init_config

cfg = init_config(overrides=[f"experiment=semantic/dales_11g"]) 
from src.transforms import instantiate_datamodule_transforms

transforms_dict = instantiate_datamodule_transforms(cfg.datamodule)

We apply this preprocessing to the tiled_data

In [9]:
# Apply pre-transforms
nag = transforms_dict['pre_transform'](tiled_data)

from src.transforms import NAGRemoveKeys
nag = NAGRemoveKeys(level=0, keys=[k for k in nag[0].keys if k not in cfg.datamodule.point_load_keys])(nag)
nag = NAGRemoveKeys(level='1+', keys=[k for k in nag[1].keys if k not in cfg.datamodule.segment_load_keys])(nag)

# Move to device
nag = nag.cuda()

# Apply on-device transforms
nag = transforms_dict['on_device_test_transform'](nag)

In [ ]:
nag.show(class_names=VANCOUVER_CLASS_NAMES, class_colors=VANCOUVER_CLASS_COLORS, keys=nag[0].keys, centroids=True, h_edge=True)

Loading the checkpoint of the SPT model trained on DALES dataset

In [10]:
import hydra 
from src.utils import init_config

ckpt_path = "../../../../mnt/efs/fs-mva/projects/superpoint_transformer/checkpoints/dales.ckpt"

cfg = init_config(overrides=[f"experiment=semantic/dales_11g"])

# Instantiate the model and load pretrained weights
model = hydra.utils.instantiate(cfg.model)
model = model._load_from_checkpoint(ckpt_path)

Apply the SPT model to our preprocessing of the raw data

In [11]:
# Set the model in inference mode on the same device as the input
model = model.eval().to(nag.device)

# Inference, returns a task-specific ouput object carrying predictions
with torch.no_grad():
    output = model(nag)

In [12]:
# Compute the level-0 (voxel-wise) semantic segmentation predictions 
# based on the predictions on level-1 superpoints and save those for 
# visualization in the level-0 Data under the 'semantic_pred' attribute
nag[0].semantic_pred = output.voxel_semantic_pred(super_index=nag[0].super_index)
nag[0].semantic_pred.shape

torch.Size([3136464])

In [ ]:
from src.datasets.dales import CLASS_NAMES as DALES_CLASS_NAMES
from src.datasets.dales import CLASS_COLORS as DALES_CLASS_COLORS

nag.show(class_names=DALES_CLASS_NAMES, class_colors=DALES_CLASS_COLORS)

In [13]:
pred = torch.argmax(nag[0].y, axis = 1)
print(pred.shape)

torch.Size([3136464])


We obtain a 70% of accuracy on the P0 voxelized point cloud!

In [14]:
torch.sum(tiled_data.y == pred)/tiled_data.y.shape[0]

tensor(0.6939, device='cuda:0')